# CS 521 HW4 — Problem 2
NetID: ws39

Run all cells in Colab with a T4 GPU. I kept the starter's data, model, attacks,
verifier and training loop. Each method uses seeds 0 and 1 and the full 10,000-image
test set. The last cell downloads the results and checkpoints from `results/`.


In [1]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]

device: cuda


In [2]:
import contextlib
import io
import json
import math
import platform
import re
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

torch.set_num_threads(2)
OUT = Path("results")
OUT.mkdir(exist_ok=True)
print(f"PyTorch {torch.__version__}; torchvision {torchvision.__version__}")
print(platform.platform(), "threads:", torch.get_num_threads())
import google.colab
assert device.type == "cuda", "Select a T4 GPU runtime first."
print("Google Colab GPU:", torch.cuda.get_device_name(0))


PyTorch 2.11.0+cu130; torchvision 0.26.0+cu130
Linux-6.6.159+-x86_64-with-glibc2.39 threads: 2
Google Colab GPU: Tesla T4


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [3]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 500kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.59MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.22MB/s]


torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [4]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [5]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [6]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [7]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

## (a) SABR and single-norm training
For each norm, I use five-step PGD within radius $(1-\lambda)\epsilon$ to choose
the center, and compute IBP cross-entropy on radius $\lambda\epsilon$ around it.
The starter fixes $\lambda=0.7$ for both norms. The first epoch uses clean loss.
For the $\ell_2$ verifier, the ball is not clipped to the pixel cube, as specified
by the starter. The PGD center itself is a valid image in [0,1].

In [8]:
LAM = {"linf": 0.7, "l2": 0.7}


def propagation_region(model, x, y, eps, lam, norm):
    tau = lam * eps
    model.eval()
    center = pgd(model, x, y, eps - tau, norm, steps=5)
    model.train()
    return center, tau


def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    center, tau = propagation_region(model, x, y, eps, lam, norm)
    z_hat = -ibp_margin_bounds(model, center, tau, norm, y)
    return F.cross_entropy(z_hat, y, reduction=reduction)


def single_norm_loss(model, x, y, t, norm):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    return certified_loss(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)

## (b) CURE loss and gradient projection
I compute both SABR bound vectors once per batch. CURE-Max takes the larger CE
loss **per image**, then averages. BA is $\mathrm{KL}(d^2\Vert d^\infty)$ on images
whose nine non-true $\ell_2$ robust logits are negative. Both distributions receive
gradients. An empty certified subset contributes zero BA.

The starter defines GP as the positive cosine similarity times the clean
gradient (not the usual orthogonal projection). I use that exact formula for
each parameter tensor, including biases. The unchanged training loop activates
it after the radius ramp, at $t\geq11$.

In [9]:
def sabr_logits(model, x, y, t):
    z = {}
    for norm in NORMS:
        center, tau = propagation_region(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)
        z[norm] = -ibp_margin_bounds(model, center, tau, norm, y)
    return z["linf"], z["l2"]


def bound_alignment(z_inf, z_2, y):
    other = torch.arange(10, device=y.device)[None, :] != y[:, None]
    certified_small = ((z_2 < 0) | ~other).all(dim=1)
    if not certified_small.any():
        return z_inf.sum() * 0.0
    log_inf = F.log_softmax(z_inf[certified_small], dim=1)
    log_2 = F.log_softmax(z_2[certified_small], dim=1)
    return (log_2.exp() * (log_2 - log_inf)).sum(dim=1).mean()


def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    z_inf, z_2 = sabr_logits(model, x, y, t)
    loss_inf = F.cross_entropy(z_inf, y, reduction="none")
    loss_2 = F.cross_entropy(z_2, y, reduction="none")
    loss = torch.maximum(loss_inf, loss_2).mean()
    if eta != 0:
        loss = loss + eta * bound_alignment(z_inf, z_2, y)
    return loss


def gradient_projection(g_n, g_c, beta):
    denom = g_n.norm() * g_c.norm()
    cosine = (g_n * g_c).sum() / denom.clamp(min=1e-12)
    projected = cosine.clamp(min=0, max=1) * g_n
    return beta * projected + (1 - beta) * g_c

## (c) My change: combine the bounds by class
Let $a^p_j=-\underline m^p_j$ be a SABR robust logit. Instead of choosing one norm
for the entire image, define $a^U_j=\max(a^\infty_j,a^2_j)$, including $a^U_y=0$.
I use
$$L_{\rm class}=\frac1n\sum_i\mathrm{CE}(a^U_i,y_i)
 +0.5\,\frac1{|\gamma|}\sum_{i\in\gamma}\mathrm{KL}(d_i^2\Vert d_i^\infty).$$
I retain the same BA and GP (beta 0.5), learning-rate schedule, small regions and
training budget as CURE-Max + BA + GP. No new hyperparameter is introduced.

Because the true entry is zero, CE is $\log(1+\sum_{j\ne y}\exp(a_j))$. Therefore
$\mathrm{CE}(a^U,y)\geq\max_p\mathrm{CE}(a^p,y)$. The inequality is strict when
different norms give worse bounds for different competing classes. For example,
vectors [0,3,-3] and [0,-3,3] each have CE about 3.05, but their classwise maximum
[0,3,3] has CE about 3.72. This objective can send gradients to both norms for one
image, so I expect fewer neglected class-specific margins. The cost is a more
conservative surrogate, which could hurt clean accuracy or optimization.

The small-region loss is a training surrogate; full-region certificates come from evaluation.

**Ablation:** replace the classwise maximum by the per-image maximum of CE while
keeping BA and GP. This is exactly CURE-Max + BA + GP from (b), so its two runs
provide the controlled ablation without repeating identical experiments.

In [10]:
def my_loss(model, x, y, t):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    z_inf, z_2 = sabr_logits(model, x, y, t)
    return (F.cross_entropy(torch.maximum(z_inf, z_2), y)
            + 0.5 * bound_alignment(z_inf, z_2, y))

### Implementation checks
Before training, I check region containment, zero-radius loss, BA masking and KL
direction, the GP formula, and the classwise loss inequality.


In [11]:
torch.manual_seed(100)
check_model = make_model()
x, y = x_train[:8], y_train[:8]
for norm in NORMS:
    eps = EPS[norm]
    center, tau = propagation_region(check_model, x, y, eps, LAM[norm], norm)
    delta = (center - x).flatten(1)
    distance = delta.abs().amax(1) if norm == "linf" else delta.norm(dim=1)
    assert (distance + tau <= eps + 1e-6).all()
    assert center.min() >= 0 and center.max() <= 1 and check_model.training
    clean = F.cross_entropy(check_model(x), y, reduction="none")
    point = certified_loss(check_model, x, y, 0.0, LAM[norm], norm, reduction="none")
    torch.testing.assert_close(point, clean, atol=1e-6, rtol=1e-5)

yi = torch.tensor([0, 0], device=device)
zi = torch.tensor([[0., -1., -2., -3., -4., -5., -6., -7., -8., -9.],
                   [0., 2., -1., -2., -3., -4., -5., -6., -7., -8.]], device=device, requires_grad=True)
z2 = torch.tensor([[0., -2., -1., -4., -3., -6., -5., -8., -7., -9.],
                   [0., 1., -2., -3., -4., -5., -6., -7., -8., -9.]], device=device, requires_grad=True)
expected = torch.distributions.kl_divergence(
    torch.distributions.Categorical(logits=z2[0]),
    torch.distributions.Categorical(logits=zi[0]))
torch.testing.assert_close(bound_alignment(zi, z2, yi), expected)
assert bound_alignment(zi[1:], z2[1:], yi[1:]).item() == 0
ga, gb = torch.autograd.grad(bound_alignment(zi, z2, yi), (zi, z2))
assert ga[0].abs().sum() > 0 and gb[0].abs().sum() > 0
assert ga[1].abs().sum() == 0 and gb[1].abs().sum() == 0

v = torch.tensor([1., 2.], device=device)
torch.testing.assert_close(gradient_projection(v, v, .5), v)
torch.testing.assert_close(gradient_projection(-v, v, .5), .5*v)
torch.testing.assert_close(gradient_projection(torch.zeros_like(v), v, .5), .5*v)
torch.testing.assert_close(gradient_projection(v, torch.zeros_like(v), .5), torch.zeros_like(v))
za = torch.tensor([[0.,3.,-3.]], device=device)
zb = torch.tensor([[0.,-3.,3.]], device=device)
target = torch.tensor([0], device=device)
assert F.cross_entropy(torch.maximum(za,zb),target) > torch.maximum(
    F.cross_entropy(za,target),F.cross_entropy(zb,target))
for fn in [cure_loss, my_loss]:
    check_model.zero_grad()
    loss = fn(check_model, x, y, 12.)
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in check_model.parameters())
print("Passed: region containment, zero-radius CE, BA, GP, union surrogate, finite gradients.")
del check_model

Passed: region containment, zero-radius CE, BA, GP, union surrogate, finite gradients.


### Running the experiments
I fix the training and evaluation seeds and use `evaluate(..., attacks=True)` with
50-step PGD on all test images. Average seconds per epoch comes from `train`.
An output observer records each epoch without changing the training loop; its first
epoch includes optimizer setup. Checkpoints and JSON results are saved in `results/`.


In [12]:
class EpochLog(io.TextIOBase):
    def __init__(self, destination, stream):
        self.destination = destination
        self.stream = stream
        self.pending = ""
        self.rows = []
        self.last = time.perf_counter()

    def write(self, text):
        self.stream.write(text)
        self.stream.flush()
        self.pending += text
        while "\n" in self.pending:
            line, self.pending = self.pending.split("\n", 1)
            match = re.match(r"epoch\s+(\d+)\s+loss\s+(\S+)", line)
            if match:
                now = time.perf_counter()
                self.rows.append(dict(epoch=int(match[1]), loss=float(match[2]), seconds=now-self.last))
                self.last = now
                self.destination.write_text(json.dumps(self.rows, indent=2))
        return len(text)

    def flush(self):
        self.stream.flush()


def run_one(name, loss_fn, seed, beta=None, epochs=20):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = make_model()
    print(f"\n{name}, seed {seed}, {epochs} epochs", flush=True)
    log = EpochLog(OUT / f"{name}_seed{seed}_history.json", sys.stdout)
    with contextlib.redirect_stdout(log):
        sec = train(model, loss_fn, seed=seed, gp_beta=beta, epochs=epochs)
    torch.save(model.state_dict(), OUT / f"{name}_seed{seed}.pt")
    # Fix evaluation RNG independently of the training method's random draws.
    torch.manual_seed(10000 + seed)
    row = evaluate(model, name, attacks=True)
    row.update(seed=seed, time=sec, epochs=epochs, training_seconds=epochs*sec)
    (OUT / f"{name}_seed{seed}.json").write_text(json.dumps(row, indent=2))
    print_results([row])
    return row


def means(rows):
    result = []
    for name in dict.fromkeys(r["name"] for r in rows):
        group = [r for r in rows if r["name"] == name]
        result.append({"name":name, "runs":len(group), **{
            k:float(np.mean([r[k] for r in group])) for k in
            ["clean","linf","l2","union","pgd_linf","pgd_l2","pgd_union","time","training_seconds"]}})
    return result


runs = []
methods_a = [("standard", standard_loss, None),
             ("linf", lambda m,x,y,t: single_norm_loss(m,x,y,t,"linf"), None),
             ("l2", lambda m,x,y,t: single_norm_loss(m,x,y,t,"l2"), None)]
methods_b = [("cure_max", lambda m,x,y,t: cure_loss(m,x,y,t,eta=0.0), None),
             ("cure_max_ba", cure_loss, None),
             ("cure_max_ba_gp", cure_loss, 0.5)]

### Results for (a)

In [13]:
for name, fn, beta in methods_a:
    for seed in (0, 1):
        runs.append(run_one(name, fn, seed, beta))
display(pd.DataFrame(means(runs)).round(3))
(OUT / "runs.json").write_text(json.dumps(runs, indent=2))


standard, seed 0, 20 epochs
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.2103  time 1 s
epoch  3  loss 0.1558  time 1 s
epoch  4  loss 0.1279  time 1 s
epoch  5  loss 0.1042  time 1 s
epoch  6  loss 0.0922  time 1 s
epoch  7  loss 0.0791  time 2 s
epoch  8  loss 0.0667  time 2 s
epoch  9  loss 0.0611  time 2 s
epoch 10  loss 0.0548  time 2 s
epoch 11  loss 0.0486  time 2 s
epoch 12  loss 0.0454  time 3 s
epoch 13  loss 0.0414  time 3 s
epoch 14  loss 0.0374  time 3 s
epoch 15  loss 0.0323  time 3 s
epoch 16  loss 0.0181  time 3 s
epoch 17  loss 0.0128  time 4 s
epoch 18  loss 0.0119  time 4 s
epoch 19  loss 0.0109  time 4 s
epoch 20  loss 0.0104  time 4 s
standard                  clean  97.50  linf   0.00  l2   0.00  union   0.00  pgd_linf   1.38  pgd_l2  10.61  pgd_union   1.37  training_seconds   4.42  time/epoch   0.2 s

standard, seed 1, 20 epochs
epoch  1  loss 0.5640  time 0 s
epoch  2  loss 0.2074  time 0 s
epoch  3  loss 0.1570  time 1 s
epoch  4  loss 0.1272  time 1 s
ep

,name,runs,clean,linf,l2,union,pgd_linf,pgd_l2,pgd_union,time,training_seconds
0,standard,2,97.480,0.000,0.000,0.000,1.145,10.02,1.140,0.209,4.173
1,linf,2,93.405,80.665,4.805,4.805,84.480,70.40,70.110,1.197,23.949
2,l2,2,92.620,54.335,58.180,51.570,69.845,78.75,69.385,1.263,25.264


1664

### Results for (b)

In [14]:
for name, fn, beta in methods_b:
    for seed in (0, 1):
        runs.append(run_one(name, fn, seed, beta))
display(pd.DataFrame(means(runs)).round(3))
(OUT / "runs.json").write_text(json.dumps(runs, indent=2))


cure_max, seed 0, 20 epochs
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4887  time 3 s
epoch  3  loss 0.5010  time 5 s
epoch  4  loss 0.5257  time 7 s
epoch  5  loss 0.5658  time 10 s
epoch  6  loss 0.6304  time 12 s
epoch  7  loss 0.7264  time 15 s
epoch  8  loss 0.8331  time 17 s
epoch  9  loss 0.9535  time 20 s
epoch 10  loss 1.0763  time 22 s
epoch 11  loss 1.2127  time 24 s
epoch 12  loss 1.2672  time 27 s
epoch 13  loss 1.2403  time 29 s
epoch 14  loss 1.2206  time 32 s
epoch 15  loss 1.2015  time 34 s
epoch 16  loss 1.1533  time 36 s
epoch 17  loss 1.1431  time 39 s
epoch 18  loss 1.1379  time 41 s
epoch 19  loss 1.1368  time 44 s
epoch 20  loss 1.1341  time 46 s
cure_max                  clean  92.24  linf  67.47  l2  57.62  union  57.53  pgd_linf  76.88  pgd_l2  77.88  pgd_union  76.10  training_seconds  46.09  time/epoch   2.3 s

cure_max, seed 1, 20 epochs
epoch  1  loss 0.5640  time 0 s
epoch  2  loss 0.4955  time 3 s
epoch  3  loss 0.5016  time 5 s
epoch  4  loss 0.5

,name,runs,clean,linf,l2,union,pgd_linf,pgd_l2,pgd_union,time,training_seconds
0,standard,2,97.480,0.000,0.000,0.000,1.145,10.02,1.140,0.209,4.173
1,linf,2,93.405,80.665,4.805,4.805,84.480,70.40,70.110,1.197,23.949
2,l2,2,92.620,54.335,58.180,51.570,69.845,78.75,69.385,1.263,25.264
3,cure_max,2,92.435,67.610,57.180,57.095,77.305,78.44,76.600,2.331,46.628
4,cure_max_ba,2,91.995,67.215,56.990,56.940,76.425,77.92,75.900,2.472,49.447
5,cure_max_ba_gp,2,92.640,66.985,56.590,56.515,76.615,78.25,76.000,2.617,52.340


3373

### Results for (c)

In [15]:
for seed in (0, 1):
    runs.append(run_one("classwise_ba_gp", my_loss, seed, beta=0.5))
averages = means(runs)
display(pd.DataFrame(averages).round(3))
(OUT / "runs.json").write_text(json.dumps(runs, indent=2))
(OUT / "means.json").write_text(json.dumps(averages, indent=2))

# If the new method is slower, give the best CURE baseline at least the same
# measured training budget. Round up to a complete epoch, in CURE's favor.
by_name = {r["name"]:r for r in averages}
best_name = max((name for name,_,_ in methods_b), key=lambda n: by_name[n]["union"])
ours = by_name["classwise_ba_gp"]
best = by_name[best_name]
matched_runs = []
if ours["training_seconds"] > best["training_seconds"]:
    matched_epochs = max(21, math.ceil(ours["training_seconds"] / best["time"]))
    _, best_fn, best_beta = next(v for v in methods_b if v[0] == best_name)
    for seed in (0, 1):
        matched_runs.append(run_one(best_name + "_time_matched", best_fn, seed,
                                    beta=best_beta, epochs=matched_epochs))
    display(pd.DataFrame(means(matched_runs)).round(3))
else:
    print("The new method was not slower than the best CURE baseline; no extra-time baseline is needed.")
summary = dict(execution_environment="Google Colab", gpu=torch.cuda.get_device_name(0), runs=runs, means=averages, best_cure=best_name, time_matched=matched_runs,
               platform=platform.platform(), device=str(device), threads=torch.get_num_threads(),
               torch_version=torch.__version__, torchvision_version=torchvision.__version__)
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))


classwise_ba_gp, seed 0, 20 epochs
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4881  time 3 s
epoch  3  loss 0.4982  time 5 s
epoch  4  loss 0.5239  time 8 s
epoch  5  loss 0.5681  time 11 s
epoch  6  loss 0.6294  time 13 s
epoch  7  loss 0.7223  time 16 s
epoch  8  loss 0.8320  time 18 s
epoch  9  loss 0.9574  time 21 s
epoch 10  loss 1.0870  time 24 s
epoch 11  loss 1.2228  time 26 s
epoch 12  loss 1.2583  time 29 s
epoch 13  loss 1.2394  time 32 s
epoch 14  loss 1.2282  time 35 s
epoch 15  loss 1.2185  time 38 s
epoch 16  loss 1.1820  time 40 s
epoch 17  loss 1.1749  time 43 s
epoch 18  loss 1.1705  time 46 s
epoch 19  loss 1.1702  time 49 s
epoch 20  loss 1.1680  time 52 s
classwise_ba_gp           clean  92.57  linf  66.98  l2  57.05  union  57.00  pgd_linf  76.18  pgd_l2  77.93  pgd_union  75.61  training_seconds  51.85  time/epoch   2.6 s

classwise_ba_gp, seed 1, 20 epochs
epoch  1  loss 0.5640  time 0 s
epoch  2  loss 0.4961  time 3 s
epoch  3  loss 0.5023  time 6 s
epoc

,name,runs,clean,linf,l2,union,pgd_linf,pgd_l2,pgd_union,time,training_seconds
0,standard,2,97.480,0.000,0.000,0.000,1.145,10.020,1.140,0.209,4.173
1,linf,2,93.405,80.665,4.805,4.805,84.480,70.400,70.110,1.197,23.949
2,l2,2,92.620,54.335,58.180,51.570,69.845,78.750,69.385,1.263,25.264
3,cure_max,2,92.435,67.610,57.180,57.095,77.305,78.440,76.600,2.331,46.628
4,cure_max_ba,2,91.995,67.215,56.990,56.940,76.425,77.920,75.900,2.472,49.447
5,cure_max_ba_gp,2,92.640,66.985,56.590,56.515,76.615,78.250,76.000,2.617,52.340
6,classwise_ba_gp,2,92.485,66.885,56.385,56.325,76.615,78.385,76.085,2.586,51.724



cure_max_time_matched, seed 0, 23 epochs
epoch  1  loss 0.5883  time 0 s
epoch  2  loss 0.4887  time 3 s
epoch  3  loss 0.5010  time 5 s
epoch  4  loss 0.5257  time 8 s
epoch  5  loss 0.5658  time 10 s
epoch  6  loss 0.6304  time 13 s
epoch  7  loss 0.7264  time 15 s
epoch  8  loss 0.8331  time 18 s
epoch  9  loss 0.9535  time 20 s
epoch 10  loss 1.0763  time 22 s
epoch 11  loss 1.2127  time 25 s
epoch 12  loss 1.2672  time 28 s
epoch 13  loss 1.2403  time 30 s
epoch 14  loss 1.2206  time 32 s
epoch 15  loss 1.2015  time 35 s
epoch 16  loss 1.1533  time 37 s
epoch 17  loss 1.1431  time 40 s
epoch 18  loss 1.1379  time 42 s
epoch 19  loss 1.1368  time 45 s
epoch 20  loss 1.1341  time 47 s
epoch 21  loss 1.1336  time 49 s
epoch 22  loss 1.1290  time 52 s
epoch 23  loss 1.1287  time 54 s
cure_max_time_matched     clean  92.35  linf  67.48  l2  57.64  union  57.53  pgd_linf  77.03  pgd_l2  78.20  pgd_union  76.34  training_seconds  54.43  time/epoch   2.4 s

cure_max_time_matched, seed 1,

,name,runs,clean,linf,l2,union,pgd_linf,pgd_l2,pgd_union,time,training_seconds
0,cure_max_time_matched,2,92.525,67.635,57.22,57.145,77.345,78.655,76.655,2.359,54.249


7458

### Per-epoch times and individual seeds
The ablation is `cure_max_ba_gp`. Compared with `classwise_ba_gp`, only the location
of the maximum in the loss changes.


In [16]:
display(pd.DataFrame(runs + matched_runs).round(3))
epoch_rows = []
for row in runs + matched_runs:
    records = json.loads((OUT / f"{row['name']}_seed{row['seed']}_history.json").read_text())
    for record in records:
        epoch_rows.append(dict(name=row['name'], seed=row['seed'], **record))
epoch_frame = pd.DataFrame(epoch_rows)
display(epoch_frame.pivot_table(index="epoch", columns="name", values="seconds").round(3))
epoch_frame.to_csv(OUT / "epoch_history.csv", index=False)
pd.DataFrame(runs + matched_runs).to_csv(OUT / "seed_results.csv", index=False)

,name,clean,linf,l2,union,pgd_linf,pgd_l2,pgd_union,seed,time,epochs,training_seconds
0,standard,97.50,0.00,0.00,0.00,1.38,10.61,1.37,0,0.221,20,4.423
1,standard,97.46,0.00,0.00,0.00,0.91,9.43,0.91,1,0.196,20,3.922
2,linf,93.52,80.88,5.51,5.51,84.58,71.45,71.19,0,1.217,20,24.333
3,linf,93.29,80.45,4.10,4.10,84.38,69.35,69.03,1,1.178,20,23.566
4,l2,93.14,54.57,58.23,51.70,68.51,78.04,68.04,0,1.273,20,25.460
5,l2,92.10,54.10,58.13,51.44,71.18,79.46,70.73,1,1.253,20,25.067
6,cure_max,92.24,67.47,57.62,57.53,76.88,77.88,76.10,0,2.304,20,46.090
7,cure_max,92.63,67.75,56.74,56.66,77.73,79.00,77.10,1,2.358,20,47.166
8,cure_max_ba,91.99,67.35,57.47,57.44,76.33,77.95,75.82,0,2.474,20,49.475
9,cure_max_ba,92.00,67.08,56.51,56.44,76.52,77.89,75.98,1,2.471,20,49.418


name,classwise_ba_gp,cure_max,cure_max_ba,cure_max_ba_gp,cure_max_time_matched,l2,linf,standard
epoch,,,,,,,,
1,0.211,0.229,0.200,0.201,0.247,0.207,0.193,0.276
2,2.511,2.595,2.677,2.718,2.531,1.367,1.189,0.196
3,2.670,2.388,2.760,2.488,2.344,1.423,1.201,0.198
4,2.763,2.378,2.484,2.502,2.474,1.343,1.261,0.196
5,2.538,2.358,2.501,2.595,2.489,1.325,1.374,0.203
6,2.450,2.451,2.481,2.726,2.449,1.247,1.284,0.214
7,2.516,2.576,2.731,2.706,2.548,1.255,1.173,0.199
8,2.660,2.448,2.808,2.511,2.358,1.262,1.196,0.197
9,2.712,2.338,2.532,2.507,2.422,1.266,1.309,0.198


## Discussion
See the report for the comparison of cross-norm transfer, BA, GP and the classwise loss.


## Sources
- Course starter: https://github.com/nalinwadhwa02/CCS521FA26HW/blob/main/hw4/q2/multi_norm_certified_training_mnist.ipynb
- Jiang and Singh (2024), *Towards Universal Certified Robustness with Multi-Norm
  Training*, original version: https://arxiv.org/abs/2410.03000v1

I follow the assignment's simplified CURE, including its specific BA direction
and per-tensor GP. The classwise maximum is the change tested in (c); the other
components are inherited from the starter and CURE.

In [18]:
import shutil
from google.colab import files
shutil.make_archive("problem2_results", "zip", "results")
files.download("problem2_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>